In [4]:
import pandas as pd

df = pd.read_csv(r"dataset\raw\ai_adoption_dataset.csv")

# Visión General
print("=== FORMA DEL DATASET ===")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}\n")

print("=== TIPOS DE DATOS Y NULOS ===")
print(df.info())
print("\n")

# Conteo de Valores Faltantes
nulos = df.isnull().sum()
print("=== VALORES FALTANTES ===")
print(nulos[nulos > 0])
print("\n")

# Valores Duplicados
print("=== DUPLICADOS ===")
print(f"Filas exactamente duplicadas: {df.duplicated().sum()}\n")

# Resumen Estadístico 
print("=== RESUMEN VARIABLES NUMÉRICAS ===")
print(df.describe().round(2))
print("\n")

print("=== VALORES ÚNICOS EN CATEGÓRICAS CLAVE ===")
cols_categoricas = ['industry', 'ai_tool', 'company_size', 'year']
for col in cols_categoricas:
    print(f"--- {col.upper()} ---")
    print(df[col].unique())
    print("\n")

=== FORMA DEL DATASET ===
Filas: 145000
Columnas: 9

=== TIPOS DE DATOS Y NULOS ===
<class 'pandas.DataFrame'>
RangeIndex: 145000 entries, 0 to 144999
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   country             145000 non-null  str    
 1   industry            145000 non-null  str    
 2   ai_tool             145000 non-null  str    
 3   adoption_rate       145000 non-null  float64
 4   daily_active_users  145000 non-null  int64  
 5   year                145000 non-null  int64  
 6   user_feedback       145000 non-null  str    
 7   age_group           145000 non-null  str    
 8   company_size        145000 non-null  str    
dtypes: float64(1), int64(2), str(6)
memory usage: 35.7 MB
None


=== VALORES FALTANTES ===
Series([], dtype: int64)


=== DUPLICADOS ===
Filas exactamente duplicadas: 0

=== RESUMEN VARIABLES NUMÉRICAS ===
       adoption_rate  daily_active_users       year
count   

In [5]:
# 1. Creamos una columna de imagen contra texto para clasificar a las IA
df_clean = pd.read_csv(r"dataset\clean\ai_adoption_dataset.csv")

# 2. Creación de la columna 'tool_category' (Texto vs Imagen)
texto_tools = ['ChatGPT', 'Claude', 'Bard']
imagen_tools = ['Midjourney', 'Stable Diffusion']

def clasificar_herramienta(tool):
    if tool in texto_tools:
        return 'Generación de Texto'
    elif tool in imagen_tools:
        return 'Generación de Imagen'
    else:
        return 'Otra'

df_clean['tool_category'] = df_clean['ai_tool'].apply(clasificar_herramienta)

# 3. Conversión de 'year' a formato texto (string/object)
df_clean['year'] = df_clean['year'].astype(str)

# 4. Verificación de las transformaciones
print("=== DISTRIBUCIÓN POR CATEGORÍA DE IA ===")
print(df_clean['tool_category'].value_counts())
print("\n=== TIPOS DE DATOS ACTUALIZADOS ===")
print(df_clean[['year', 'tool_category']].dtypes)

=== DISTRIBUCIÓN POR CATEGORÍA DE IA ===
tool_category
Generación de Texto     79850
Generación de Imagen    65150
Name: count, dtype: int64

=== TIPOS DE DATOS ACTUALIZADOS ===
year             str
tool_category    str
dtype: object


In [ ]:

import matplotlib.pyplot as plt
# ==========================================
# EDA 1: Adopción vs. Tamaño de Empresa
# ==========================================
import seaborn as sns

sns.set_theme(style='whitegrid', palette='deep')

company_size_order = ['Startup', 'SME', 'Enterprise']

print("=== EDA 1: Tasa de adopción promedio por tamaño de empresa ===")
eda_1 = (
    df_clean.groupby('company_size', observed=True)['adoption_rate']
    .agg(['mean', 'median', 'count'])
    .reindex(company_size_order)
    .round(2)
)
print(eda_1)

eda_1_plot = eda_1.reset_index()

plt.figure(figsize=(9, 5))
ax = sns.barplot(
    data=eda_1_plot,
    x='company_size',
    y='mean',
    order=company_size_order,
    color='#4C72B0',
    ci=None,
)

for index, value in enumerate(eda_1_plot['mean']):
    ax.text(index, value + 1, f'{value:.2f}%', ha='center', fontweight='bold')

ax.set_title('Adopción promedio por tamaño de empresa')
ax.set_xlabel('Tamaño de empresa')
ax.set_ylabel('Tasa de adopción promedio (%)')
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()
print("\n" + "-"*50 + "\n")

# ==========================================
# EDA 2: Texto vs. Imagen por Industria
# ==========================================
print("=== EDA 2: Adopcion promedio de IA de texto VS IA de imagen por industria ===")
# ¿Qué queremos averiguar?: ¿ChatGPT/Texto es transversal a todas las industrias y Midjourney/Imagen tiene nichos específicos? (Hipótesis 2)
eda_2 = pd.pivot_table(
    df_clean, 
    values='adoption_rate', 
    index='industry', 
    columns='tool_category', 
    aggfunc='mean'
).round(2)

# Calculamos la diferencia entre ambos para ver que tecnología predomina en cada rubro
eda_2['Diferencia (Texto - Imagen)'] = eda_2['Generación de Texto'] - eda_2['Generación de Imagen']
eda_2 = eda_2.sort_values(by='Diferencia (Texto - Imagen)', ascending=False)
print(eda_2)

=== EDA 1: Tada de adopcion por tamañano de la empresa ===
               mean  median  count
company_size                      
Startup       50.04   49.98  48601
Enterprise    49.84   49.68  48279
SME           49.74   49.62  48120

--------------------------------------------------

=== EDA 2: Adopcion promedio de IA de texto VS IA de imagen por industria ===
tool_category   Generación de Imagen  Generación de Texto  \
industry                                                    
Transportation                 49.52                50.10   
Finance                        49.65                50.22   
Healthcare                     49.72                49.99   
Agriculture                    50.21                50.42   
Manufacturing                  49.62                49.54   
Retail                         49.66                49.56   
Technology                     50.11                49.95   
Education                      50.06                49.57   

tool_category   Diferenc